# 黑马头条项目｜第08天：平台管理实战

> 学习目标：为黑马头条补充运营管理端，完成管理员登录与网关、频道管理、敏感词管理、用户实名认证审核和自媒体文章人工审核。

本篇依据百度网盘第08天的《平台管理-需求说明.md》和《平台管理-接口文档.md》网页预览整理。未读取参考代码、前端 `admin-web` 文件夹或数据库脚本，未启动 Nginx、Nacos、MySQL、Docker、微服务或第三方公安/活体认证接口，未执行任何真实身份证、照片、密码或令牌操作。接口示例中的地址、路径和凭据均以占位符表示，资料中出现的示例 JWT 不在笔记中复现。

第08天是一个多人协作的实战任务：资料要求以第07天代码为基础创建共享 Git 项目，组员各自建立分支开发，最后合并并准备答辩。这里记录的是业务、接口和工程设计，不能代替源码实现或联调结果。


## 1. 平台管理端的角色与边界

平台管理端面向运营管理员，负责内容和用户的后台管理。APP用户、自媒体作者、article内容服务与admin平台的职责不同：

| 角色/服务 | 主要职责 |
|---|---|
| APP用户 | 实名认证申请、浏览和互动 |
| 自媒体端 | 作者提交文章、自动审核、文章上下架 |
| article服务 | APP文章数据、静态详情和展示配置 |
| admin平台 | 频道、敏感词、认证和人工审核 |
| admin网关 | JWT校验、路由和统一入口 |

```text
admin-web
    │ HTTP
    ▼
admin-gateway（JWT校验、路由）
    ├── admin服务：登录、频道、敏感词、认证审核
    ├── wemedia服务：人工文章审核
    └── 其他只读/业务服务
```

管理员端能改变用户认证、文章审核和频道状态，属于高权限后台。除登录外，所有接口都应校验管理员身份、角色权限、操作对象和状态条件；网关鉴权不能替代服务内部的授权检查。


## 2. 团队协作与工程基线

资料要求：

1. 组长创建一个开源 Git 项目；
2. 基础代码使用第07天完成后的代码；
3. 每位组员克隆同一仓库；
4. 每位组员创建个人分支完成功能；
5. 通过合并请求整合为完整项目；
6. 展示各项功能并准备答辩。

建议把工作拆为可合并的垂直模块：

| 分支任务 | 交付范围 |
|---|---|
| `admin-auth` | 管理员登录、JWT、网关鉴权 |
| `channel` | 频道增删改查和引用校验 |
| `sensitive` | 敏感词增删改查、唯一性 |
| `user-auth-review` | 用户认证列表、通过/拒绝、账号开通 |
| `news-review` | 待人工文章列表、详情、拒绝/通过 |
| `admin-web` | 页面、代理和联调 |

每个分支提交前应统一DTO、响应码、分页模型、异常处理和数据库迁移脚本。合并顺序优先公共模型与网关，再合并业务模块，减少接口字段漂移。用户没有要求创建仓库或执行 Git 命令，本篇只整理资料要求。


## 3. admin-web 与 Nginx 入口

资料要求使用 `admin-web` 文件夹中的前端并通过 Nginx 部署。逻辑上有两个入口：

- `/` 返回后台前端静态文件；
- `/service_<GATEWAY_PORT>/...` 反向代理到平台网关，前端不直接访问各微服务。

配置结构示意（将课程固定路径和端口替换为占位符）：

```nginx
upstream heima-admin-gateway {
    server <ADMIN_GATEWAY_HOST>:<ADMIN_GATEWAY_PORT>;
}

server {
    listen <ADMIN_WEB_PORT>;
    location / {
        root <ADMIN_WEB_ROOT>;
        index index.html;
    }
    location ~ /service_<ADMIN_GATEWAY_PORT>/(.*) {
        proxy_pass http://heima-admin-gateway/$1;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
    }
}
```

生产上要明确前端路由回退、静态资源缓存、HTTPS、CORS、请求体大小和上传图片策略。正则代理路径要避免意外吞掉前缀或开放未授权的内部接口。Nginx只负责入口和代理，不能将它当作管理员权限校验。


## 4. 管理员登录与密码安全

接口文档：`POST /login/in`，请求体包含 `name` 和 `password`，返回 `ResponseResult`，成功数据包含管理员信息和 token。资料需求强调密码需要手动加盐验证。

```json
{
  "name": "<ADMIN_NAME>",
  "password": "<PASSWORD>"
}
```

推荐登录流程：

```text
输入用户名/密码
       ↓
查询管理员（只取必要字段）
       ↓
读取每个账号独立salt
       ↓
使用慢哈希校验密码
       ↓
生成短时JWT + 刷新/重新登录策略
       ↓
返回最小用户信息和token
```

课程的“手动加盐”是学习点，生产不要使用简单 `MD5(password + salt)` 或可逆加密，优先使用 BCrypt、Argon2 等带成本参数的密码哈希。登录失败信息要避免泄露“用户不存在”与“密码错误”的账号枚举差异；接口文档示例中出现的用户资料、邮箱、电话和JWT属于示例敏感数据，笔记不复现。

登录响应不应返回密码、salt或完整管理员对象。token不写入日志，前端使用安全存储，网关和业务服务校验签名、过期时间、发行者、权限声明，并防止重放。退出、禁用账号或密钥轮换时需要使旧token失效或缩短有效期。


## 5. admin网关：JWT校验与服务路由

平台后台所有请求经过admin网关，网关主要做：

1. 识别登录白名单，例如 `/login/in`；
2. 从 `Authorization: Bearer <TOKEN>` 读取JWT；
3. 校验签名、过期和必要声明；
4. 将可信管理员身份传给下游（避免信任客户端自填Header）；
5. 按路径路由到admin、wemedia等微服务；
6. 统一处理未认证、无权限、服务不可用和跨域策略。

```yaml
spring:
  cloud:
    gateway:
      routes:
        - id: leadnews-admin
          uri: lb://leadnews-admin
          predicates:
            - Path=/admin/**
        - id: leadnews-wemedia-review
          uri: lb://leadnews-wemedia
          predicates:
            - Path=/wemedia/**
```

实际路由前缀应与前端代理和服务Controller的 `@RequestMapping` 对齐，不能照搬示意。网关校验失败直接返回401；token有效但权限不足返回403。服务端仍要检查管理员角色和资源归属，防止绕过网关直连内部地址。


## 6. 频道管理：业务规则

频道用于文章分类。需求规则：

- 新增时频道名称不能重复；
- 列表按创建时间倒序；
- 支持名称模糊查询、状态精确查询和分页；
- 编辑时可以修改频道；被文章引用的频道不能禁用；
- 只有已禁用频道才能删除。

```text
新建 ──► 启用 ──► 禁用 ──► 删除
          │        ▲
          └─被文章引用时不能直接禁用
```

状态在接口文档中使用boolean：`true`表示启用，`false`表示禁用；需求文字也写成1/0。服务内部应统一一种类型并在DTO边界转换，避免把JSON `true`、数字1和字符串`"1"`混为一谈。

禁用前应查询文章引用关系，或用带引用条件的更新并检查影响行数。删除是可恢复还是物理删除要先定策略；若存在审计、历史文章或外键关系，通常采用软删除或仅禁用。名称唯一要在服务校验之外建立数据库唯一约束，处理并发新增。


## 7. 频道接口清单与分页

接口文档列出：

| 功能 | 方法 | 路径 | 关键参数 |
|---|---|---|---|
| 删除频道 | GET（原文档约定） | `/api/v1/channel/del/{id}` | path id |
| 列表 | POST | `/api/v1/channel/list` | name、page、size |
| 新增 | POST | `/api/v1/channel/save` | name、description、ord、status等 |
| 修改 | POST | `/api/v1/channel/update` | id及修改字段 |

删除会产生状态变化，按HTTP语义更建议使用 `DELETE /api/v1/channel/{id}`；如果为了兼容课程前端保留GET，应通过网关限制、审计并避免被预取或缓存误触发。该建议不改变资料中原路径的记录。

分页请求示意：

```json
{
  "name": "<OPTIONAL_NAME>",
  "status": true,
  "page": 1,
  "size": 10
}
```

接口文档的频道列表示例记录了name/page/size，需求额外要求status精确查询；实现时要补齐字段或明确以实际DTO为准。页码、每页大小应校验上下限，排序固定为 `createdTime DESC, id DESC`，让同一时间的结果稳定。


## 8. 敏感词管理与审核词库

敏感词管理为第04天自动审核提供自管理词库。需求：新增敏感词不可重复；列表按创建时间倒序，支持敏感词名称模糊查询和分页；支持修改和直接删除。

接口文档路径：

| 功能 | 方法 | 路径 |
|---|---|---|
| 删除 | DELETE | `/api/v1/sensitive/del/{id}` |
| 列表 | POST | `/api/v1/sensitive/list` |
| 保存 | POST | `/api/v1/sensitive/save` |
| 修改 | POST | `/api/v1/sensitive/update` |

```json
{
  "sensitives": "<SENSITIVE_WORD>",
  "page": 1,
  "size": 10
}
```

词条保存前应trim、规范化Unicode和长度，数据库建立唯一约束。修改时要防止把词改成已存在的另一词。删除后，审核服务的DFA或缓存词库不能继续永久使用旧值，应通过版本、发布事件或定时刷新让词库更新可追踪。

敏感词本身属于审核规则，不应在普通日志中明文打印；管理操作要记录谁在何时新增、修改或删除了哪一个版本。接口返回可显示词条，但只给授权管理员。


## 9. 用户实名认证审核：数据与状态

需求中的认证材料包括姓名、身份证号、身份证正反面、手持照片和活体动作。用户提交后进入运营审核，审核包括身份核验（资料要求对接公安系统），审核通过后开通自媒体账号，用户名和密码与APP账号一致。

```text
APP提交认证材料
        ↓
待审核记录
        ├─ 身份/材料校验失败 → 拒绝并记录原因
        ├─ 人工补充核验       → 保持待处理/人工状态
        └─ 审核通过           → 更新认证状态 + 开通自媒体账号
```

身份证和人脸材料是高度敏感个人信息：

- 最小化收集、加密传输和加密存储，限制管理员可见范围；
- 展示时脱敏身份证号、照片和日志；
- 第三方核验设置超时、重试上限、访问审计与数据保留期限；
- 开通自媒体账号时不要把明文密码复制到日志或消息；
- 账号开通和认证状态更新需要幂等，避免重复创建账号。

本篇只记录业务需求，不执行公安接口、活体检测或真实材料处理。


## 10. 认证审核接口

接口文档列出：

| 功能 | 方法 | 路径 | 参数 |
|---|---|---|---|
| 查询认证列表 | POST | `/api/v1/auth/list` | id、msg、status、page、size |
| 审核失败 | POST | `/api/v1/auth/authFail` | AuthDto，包含id/msg等 |
| 审核成功 | POST | `/api/v1/auth/authPass` | AuthDto，包含id/msg等 |

```json
{
  "id": 123,
  "status": 0,
  "msg": "<OPTIONAL_REASON>",
  "page": 1,
  "size": 10
}
```

服务层应再次校验当前状态是否允许转换，使用条件更新避免两个管理员同时审核：

```text
UPDATE auth
SET status = PASS, reason = ?, reviewed_by = ?, reviewed_at = ?
WHERE id = ? AND status = PENDING
```

更新影响行数为0时返回“已处理/状态冲突”，而不是重复执行开账号。认证通过与自媒体账号开通跨表、跨服务时，可使用事务边界、Outbox或补偿任务，记录“认证通过但账号开通待补偿”状态。拒绝原因应可审计并向用户展示适当的非敏感说明。


## 11. 自媒体文章人工审核：来源与状态

第04天自动审核遇到不确定内容时，文章状态为3，进入平台人工审核。admin端要：

- 分页查询待人工文章；
- 按标题模糊查询、按审核状态精确查询；
- 按创建时间倒序；
- 展示作者名称、文章详情和素材；
- 违规时驳回，状态改为2并填写原因；
- 合规时创建APP文章，更新自媒体文章状态。

```text
自媒体 wm_news.status = 3（人工审核）
             │
             ▼
admin 查看列表/详情
       ┌─────┴─────┐
       ▼           ▼
    拒绝         审核通过
 status=2      创建APP文章
 reason=...    回写articleId/status
```

通过操作应复用第04天的文章保存和静态化逻辑，不能只把状态改成“通过”而漏掉APP文章和内容记录。人工审核前再次读取最新文章状态和内容，防止作者改稿、撤稿或自动任务已处理后仍显示旧页面。


## 12. 文章列表与详情接口

接口文档：

| 功能 | 方法 | 路径 | 参数 |
|---|---|---|---|
| 文章列表 | POST | `/api/v1/news/list_vo` | NewsAuthDto：id、msg、page、size、status、title |
| 文章详情 | GET | `/api/v1/news/one_vo/{id}` | path id |
| 审核失败 | POST | `/api/v1/news/auth_fail` | NewsAuthDto：id、msg |
| 审核成功 | POST | `/api/v1/news/auth_pass` | NewsAuthDto：id、msg |

列表条件示意：

```json
{
  "title": "<OPTIONAL_TITLE>",
  "status": 3,
  "page": 1,
  "size": 10
}
```

数据库查询按 `created_time DESC, id DESC`，标题使用受控模糊查询，状态使用精确匹配，分页大小设置上限。列表VO要通过作者ID关联或批量查询作者名称，避免逐行查询造成N+1；详情接口也应展示作者名称。

接口文档的示例响应含文章正文JSON和图片地址，笔记不复制其中的真实路径。返回详情时校验管理员权限，并对富文本、图片URL和外链做安全处理；不要把不必要的作者隐私字段返给前端。


## 13. 人工驳回：状态、原因与幂等

拒绝接口：`POST /api/v1/news/auth_fail`。规则是必须给出原因，文章状态改为2（审核失败）。

```java
public ResponseResult authFail(NewsAuthDto dto) {
    if (dto == null || dto.getId() == null
            || StringUtils.isBlank(dto.getMsg())) {
        return ResponseResult.errorResult(PARAM_INVALID);
    }
    int affected = newsMapper.rejectIfManualReview(
            dto.getId(), dto.getMsg(), currentAdminId());
    if (affected != 1) {
        return ResponseResult.errorResult(STATE_CONFLICT);
    }
    return ResponseResult.okResult(SUCCESS);
}
```

这是学习示意，具体枚举和Mapper名称按工程统一。条件更新应只允许 `status=3` 的文章被拒绝；重复点击不能把已发布文章改回失败。原因长度、敏感信息和可展示内容应有校验，审计日志记录管理员、时间、旧状态和新状态。

驳回后如果文章仍有延迟任务或索引事件，任务执行入口需重新检查状态，搜索索引也要按第07天的同步方案下架或过滤，否则用户可能仍能看到旧文章。


## 14. 人工通过：创建APP文章与跨服务一致性

通过接口：`POST /api/v1/news/auth_pass`。需求要求：通过审核后创建APP端文章，并更新自媒体文章状态。

推荐流程：

```text
条件读取 wm_news(status=3)
        ↓
锁定/版本校验，防止重复审核
        ↓
调用 article 服务保存 ApArticle、Config、Content
        ↓
成功返回 articleId
        ↓
回写 wm_news.article_id，status=8/9（按项目状态约定）
        ↓
触发静态化、ES索引等后续事件
```

第04天的状态表中 `8` 是审核通过待发布，`9` 是已发布；平台需求只说“更新自媒体文章状态”，实际取值必须和项目状态机、发布时间规则一致，不能盲写数字。若文章是未来发布，审核通过与可见时间要按第05天调度设计区分。

article保存和wemedia回写不在同一个本地事务。要处理：

- article已保存但wemedia回写失败：通过articleId关联记录重试回写；
- wemedia状态已改但article保存失败：保持待补偿状态，不标记已发布；
- 管理员重复点击：article保存接口使用业务幂等键（wmNewsId/版本），返回已有articleId；
- 下游超时：不能直接重试未知结果，先查询或使用幂等请求。

通过后还要让搜索索引、上下架配置和静态页状态最终一致。本篇未执行Feign、数据库或Kafka调用。


## 15. 状态机与权限检查清单

### 15.1 认证状态

```text
待审核 ──► 审核通过 ──► 自媒体账号已开通
   └──► 审核拒绝
```

只允许从当前待审核状态转换，账号开通必须幂等。是否允许重新提交、补充材料、撤回申请由产品明确，不能通过任意状态覆盖。

### 15.2 文章审核状态

```text
1 待审核 → 自动审核 → 2失败 / 3人工 / 8待发布 / 9已发布
3人工   → 人工拒绝2 / 人工通过8或9
```

### 15.3 每个管理接口都要检查

- JWT有效、管理员角色和操作权限；
- DTO必填字段、长度、枚举和分页上限；
- 目标记录存在且属于允许操作的状态；
- 版本/条件更新影响行数；
- 关联数据约束（频道引用、作者、文章、账号）；
- 重要变更审计、异常响应和幂等键；
- 返回最小必要字段，隐藏密码、salt、身份证和原始照片。

权限校验不能只由前端按钮控制，也不能只依赖网关Header。服务层应从认证上下文读取管理员身份，禁止客户端传入 `reviewedBy`、`userId` 来冒充操作人。


## 16. 接口文档中的实现注意点

资料接口文档更像Swagger导出，存在一些需要落地时确认的地方：

| 资料现象 | 风险 | 实现时处理 |
|---|---|---|
| 登录响应示例包含完整用户对象和JWT | 暴露敏感示例并诱导日志复制 | 只返回最小字段，token脱敏；笔记不复现 |
| 频道删除路径使用GET | 具有副作用，可能被预取/缓存误触发 | 兼容时限流审计，优先DELETE |
| 频道需求有状态筛选，列表参数示例未展示 | 前后端字段不一致 | 统一DTO并增加契约测试 |
| 审核接口复用含page/size的DTO | 审核动作语义不清 | 动作接口只接受id/reason/version |
| 文章成功只写“创建APP文章” | 跨服务失败可能半成功 | 幂等、Outbox、补偿和状态记录 |
| 认证需要公安/活体接口 | 高敏感数据和外部依赖 | 最小化、超时、审计、人工兜底 |
| 需求提到数据库脚本、前端和资料文件 | 未读取不能声称源码已验证 | 本篇明确来源和限制 |

接口文档的响应码含有200/201/401/403/404，项目统一的 `ResponseResult.code` 还应与HTTP状态码约定清楚。错误信息不要泄露密码校验细节、身份证核验细节或内部服务地址。


## 17. 测试与联调计划（本次未执行）

### 登录与网关

1. 正确账号密码：返回token和最小用户信息；
2. 错误密码/不存在账号：统一安全错误；
3. 无token、过期token、错误签名：网关返回401；
4. 有效token但无权限：返回403；
5. 绕过网关访问服务：内网策略或服务鉴权拒绝。

### 频道与敏感词

1. 重复频道名/敏感词：数据库约束和友好错误；
2. 频道名称、敏感词模糊查询和分页；
3. 被引用频道禁止禁用；禁用频道才能删除；
4. 敏感词修改为已存在词时拒绝；
5. 词库刷新后自动审核使用新版本。

### 认证与文章审核

1. 待审核认证按状态分页，拒绝必须有原因；
2. 认证通过只开通一次自媒体账号；
3. 文章状态3可列出并显示作者和详情；
4. 驳回后状态2、原因和审计信息正确；
5. 通过后APP文章、wm_news.article_id、静态页和索引最终一致；
6. 双管理员重复点击只有一个成功；
7. 下游超时和重试不产生重复APP文章。

排查顺序：先确认前端代理与网关路由，再查JWT和权限，随后查请求DTO/状态条件、数据库影响行数，最后查Feign、消息、静态化和索引补偿。每个跨服务阶段都要有可关联的requestId、业务ID和审计日志。


## 18. 复习与自测

### 核心主线

- admin-web通过Nginx进入admin网关，网关负责JWT校验和路由，服务端继续做权限检查。
- 频道管理要保证名称唯一、状态查询、引用约束和禁用后删除。
- 敏感词管理是第04天自管理审核词库的运营入口，唯一性和词库刷新同样重要。
- 实名认证涉及极高敏感信息；通过后开通自媒体账号，必须幂等、审计和最小化暴露。
- 状态3文章进入人工审核；拒绝到2并记录原因，通过要创建APP文章并回写自媒体状态。
- 跨服务成功不是一个本地事务，需幂等、条件更新、可靠事件和补偿。

### 自测题

1. **为什么管理员登录后还要在业务服务检查权限？** 网关可能被绕过、路由配置可能变化；资源级授权必须由真正修改数据的服务执行。
2. **频道为什么不能只用代码判断重名？** 并发请求可能同时通过判断，数据库唯一约束才能提供最终保护。
3. **为什么被文章引用的频道不能禁用？** 文章仍依赖其分类，禁用可能破坏展示、发布和运营筛选规则，应有迁移或明确的引用策略。
4. **认证通过和账号开通为什么要幂等？** 重试、重复点击或消息重复可能多次执行，幂等键确保只创建一个账号。
5. **文章人工审核通过后为什么不能只更新status？** APP文章、内容配置、静态页和索引可能都还不存在，用户无法正常查看；需复用发布链路。
6. **如何避免旧审核结果覆盖新文章状态？** 使用状态条件、版本号或乐观锁，审核前重新读取最新记录。
7. **为什么身份证照片不能放普通日志？** 它们是高度敏感个人信息，日志扩散范围大且难以撤回，应脱敏、加密和限制访问。
8. **接口文档中的GET删除能否直接照搬？** 这是资料的原约定，但有副作用；更安全的设计使用DELETE，兼容旧前端时要加入审计和防误触发措施。

资料需求末尾重点是团队分工和答辩，本篇将接口、状态与工程边界补成复习材料，未把未读取的源码或前端行为写成已验证事实。
